# Audio Fundamentals: Waveforms, STFT Spectrograms, Mel Filterbanks & MFCCs

This interactive lab covers:
1. **Waveforms & Sampling**: Discrete audio signal representation and the Nyquist-Shannon sampling theorem.
2. **Short-Time Fourier Transform (STFT)**: Windowing (Hann), frame hop, and power spectrograms.
3. **Perceptual Mel Scale**: Warping linear frequencies to mimic human cochlear non-linear frequency resolution.
4. **Log-Mel Spectrograms & MFCCs**: Discrete Cosine Transform (DCT) decorrelation for cepstral analysis.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('05-speech-audio/audio-fundamentals/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import numpy as np
from audio_dsp import (
    generate_synthetic_tone,
    compute_stft_spectrogram,
    amplitude_to_db,
    create_mel_filterbank,
    compute_mel_spectrogram,
    compute_mfcc,
)

print('Audio DSP & Feature Extraction engine loaded successfully!')

## 1. Waveforms & Discrete Sampling

Synthesizing a 0.5-second acoustic wave combining fundamental tone A4 (440 Hz) and its octave harmonic A5 (880 Hz).

In [ ]:
sr = 16000  # 16 kHz sampling rate
duration = 0.5
t, signal = generate_synthetic_tone(frequencies=(440.0, 880.0), duration=duration, sample_rate=sr)

print(f'Total samples:      {len(signal):,}')
print(f'Duration:           {duration} seconds')
print(f'Signal amplitude:   [{signal.min():.3f}, {signal.max():.3f}]')
print(f'Nyquist frequency:  {sr / 2.0} Hz')

## 2. Short-Time Fourier Transform (STFT) & Power Spectrogram

Segmenting the continuous waveform into short overlapping windows (win_length=400, hop=160, n_fft=512).

In [ ]:
freqs, times, power_spec = compute_stft_spectrogram(signal, n_fft=512, hop_length=160, win_length=400)
db_spec = amplitude_to_db(power_spec)

print(f'Linear Spectrogram shape (FFT bins x Time frames): {power_spec.shape}')
print(f'Decibel Spectrogram range (dB):                    [{db_spec.min():.2f}, {db_spec.max():.2f}] dB')

## 3. Mel Filterbanks & Log-Mel Spectrogram

Multiplying linear spectral energy by 40 triangular Mel filterbanks covering [0, 8000] Hz.

In [ ]:
filterbank = create_mel_filterbank(sample_rate=sr, n_fft=512, num_mel_bins=40)
mel_spec = compute_mel_spectrogram(signal, sample_rate=sr, n_fft=512, hop_length=160, num_mel_bins=40)

print(f'Mel Filterbank matrix shape:   {filterbank.shape}')
print(f'Log-Mel Spectrogram shape:     {mel_spec.shape}')
print(f'Log-Mel energy range:          [{mel_spec.min():.2f}, {mel_spec.max():.2f}] dB')

## 4. Mel-Frequency Cepstral Coefficients (MFCCs)

Applying Discrete Cosine Transform (DCT-II) along the filterbank dimension to retain the top 13 cepstral coefficients.

In [ ]:
mfcc = compute_mfcc(mel_spec, num_mfcc=13)

print(f'MFCC feature matrix shape:     {mfcc.shape}')
print(f'MFCC 0 (overall energy) mean:  {mfcc[0].mean():.2f}')
print(f'MFCC 1 (spectral slope) mean:  {mfcc[1].mean():.2f}')
assert mfcc.shape == (13, mel_spec.shape[1])